# Additional Task A — Tolstoy vs. Dostoevsky: Vocabulary Analysis

## Preliminaries

### Installation

Install BeautifulSoup for HTML parsing and scikit-learn for TF-IDF. All other dependencies are shared with the main assignment.

In [ ]:
!pip install beautifulsoup4 requests scikit-learn scipy numpy matplotlib pandas --quiet

### Imports

Core imports for the full task.

In [ ]:
import re
import time
from collections import Counter
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from scipy.stats import anderson
from sklearn.feature_extraction.text import TfidfVectorizer

plt.rcParams['figure.dpi'] = 120

---

## A.1 Data Acquisition

### Fetching Texts from az.lib.ru

Both novels are served in Windows-1251 (CP-1251) encoding. `fetch_azlib` scans the author's index page for anchor links whose visible text contains any of the supplied keywords, then downloads the linked page and extracts plain text with BeautifulSoup. `urljoin` resolves relative hrefs correctly regardless of the site's directory structure.

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; educational-research)'}


def fetch_azlib(author_url, keywords):
    resp = requests.get(author_url, headers=HEADERS, timeout=30)
    resp.encoding = 'cp1251'
    soup = BeautifulSoup(resp.text, 'html.parser')
    for a in soup.find_all('a', href=True):
        anchor = a.get_text(strip=True).lower()
        if any(kw in anchor for kw in keywords):
            href = urljoin(author_url, a['href'])
            page = requests.get(href, headers=HEADERS, timeout=180)
            page.encoding = 'cp1251'
            body = BeautifulSoup(page.text, 'html.parser')
            return body.get_text(separator=' ')
    raise ValueError(f"No link matching {keywords!r} on {author_url}")

In [ ]:
TOLSTOY_URL = 'http://az.lib.ru/t/tolstoj_lew_nikolaewich/'
DOSTOEVSKY_URL = 'http://az.lib.ru/d/dostoewskij_f_m/'

raw_tolstoy = fetch_azlib(TOLSTOY_URL, ['война и мир'])
time.sleep(2)
raw_dostoevsky = fetch_azlib(DOSTOEVSKY_URL, ['братья карамазов'])

print(f"Tolstoy raw chars:    {len(raw_tolstoy):,}")
print(f"Dostoevsky raw chars: {len(raw_dostoevsky):,}")

---

## A.2 Preprocessing

### Tokenization

Apply the regex pattern `[а-яёa-z]+` to the lowercased text. This extracts Cyrillic tokens (including ё) and residual Latin, while discarding digits, punctuation, and all HTML artifacts. No stemming or stopword removal is applied — raw tokens only, reusing the same pipeline as Task 2 of the main assignment.

In [ ]:
def tokenize_ru(text):
    return re.findall(r'[а-яёa-z]+', text.lower())


tokens_tolstoy = tokenize_ru(raw_tolstoy)
tokens_dostoevsky = tokenize_ru(raw_dostoevsky)

print(f"Tolstoy tokens:    {len(tokens_tolstoy):,}")
print(f"Dostoevsky tokens: {len(tokens_dostoevsky):,}")

---

## A.3 Heaps' Law Analysis

### Vocabulary Growth Curves

Track cumulative vocabulary size at regular sampling points. `n_points=1000` gives a smooth curve regardless of corpus size while keeping the array small for plotting and fitting. The random seed `42` via `numpy.random.default_rng` ensures reproducible shuffles.

In [ ]:
def vocab_growth(tokens, n_points=1000):
    n = len(tokens)
    step = max(1, n // n_points)
    vocab = set()
    sizes, counts = [], []
    for i, token in enumerate(tokens, 1):
        vocab.add(token)
        if i % step == 0:
            sizes.append(len(vocab))
            counts.append(i)
    if not sizes or counts[-1] != n:
        sizes.append(len(vocab))
        counts.append(n)
    return np.array(counts, dtype=float), np.array(sizes, dtype=float)

In [ ]:
rng = np.random.default_rng(42)

tol_shuf = rng.permutation(tokens_tolstoy).tolist()
dos_shuf = rng.permutation(tokens_dostoevsky).tolist()

x_to, y_to = vocab_growth(tokens_tolstoy)
x_ts, y_ts = vocab_growth(tol_shuf)
x_do, y_do = vocab_growth(tokens_dostoevsky)
x_ds, y_ds = vocab_growth(dos_shuf)

### Power-Law Fitting

Fit $\log V = \beta \log T + \log K$ via `numpy.polyfit` in log–log space. The slope gives $\beta$ and the intercept recovers $\log K$. Standard OLS in log-space is the classical estimator for Heaps' law parameters and is consistent with the fitting convention from the main assignment.

In [ ]:
def fit_heaps(x, y):
    lx, ly = np.log(x), np.log(y)
    coef = np.polyfit(lx, ly, 1)
    beta, log_K = coef
    K = np.exp(log_K)
    residuals = ly - np.polyval(coef, lx)
    return K, beta, coef, lx, ly, residuals


curves = [
    ('Tolstoy — original',    x_to, y_to, 'steelblue',  '-'),
    ('Tolstoy — shuffled',    x_ts, y_ts, '#94b9d5',    '--'),
    ('Dostoevsky — original', x_do, y_do, 'darkorange', '-'),
    ('Dostoevsky — shuffled', x_ds, y_ds, '#f5c18a',    '--'),
]

fit_heaps_params = {}
print(f"{'Curve':<30s}  {'K':>10s}  {'β':>10s}")
print('-' * 55)
for label, x, y, *_ in curves:
    K, beta, coef, lx, ly, res = fit_heaps(x, y)
    fit_heaps_params[label] = {
        'K': K, 'beta': beta, 'coef': coef,
        'lx': lx, 'ly': ly, 'residuals': res,
    }
    print(f"{label:<30s}  {K:>10.4f}  {beta:>10.4f}")

Plot all four curves with their fitted power-law lines (dotted).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for label, x, y, color, ls in curves:
    fp = fit_heaps_params[label]
    y_fit = np.exp(np.polyval(fp['coef'], fp['lx']))
    ax.loglog(x, y, color=color, ls=ls, linewidth=1.6, label=label)
    ax.loglog(x, y_fit, color=color, ls=':', linewidth=0.9, alpha=0.7)

ax.set_xlabel('Total tokens (log scale)')
ax.set_ylabel('Vocabulary size (log scale)')
ax.set_title("Heaps' Law — Original vs. Shuffled Token Order")
ax.legend(loc='upper left')
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

### Anderson-Darling Test

**Why Anderson-Darling, not Kolmogorov-Smirnov:** AD assigns greater weight to the tails of the distribution — precisely where power-law behavior (heavy tail) is most theoretically meaningful. KS is maximally sensitive near the median, which is less relevant for laws whose entire claim is about tail structure. AD is also uniformly more powerful for continuous distributions.

We test whether the residuals of the log–log linear fit are normally distributed. Normally distributed residuals indicate an adequate power-law fit. H₀: residuals ∼ Normal(0, σ²). We report the AD statistic and 5 % critical value; rejecting H₀ means the power law is a poor description of the growth curve.

In [ ]:
print(f"{'Curve':<30s}  {'AD stat':>8s}  {'5% CV':>8s}  Reject H₀?")
print('-' * 62)

ad_heaps_results = {}
for label, *_ in curves:
    residuals = fit_heaps_params[label]['residuals']
    ad_res = anderson(residuals, dist='norm')
    idx = np.argmin(np.abs(np.array(ad_res.significance_level) - 5.0))
    reject = ad_res.statistic > ad_res.critical_values[idx]
    ad_heaps_results[label] = ad_res
    flag = 'yes' if reject else 'no'
    print(f"{label:<30s}  {ad_res.statistic:>8.4f}  "
          f"{ad_res.critical_values[idx]:>8.4f}  {flag}")

**Interpretation.** If the AD statistic falls below the 5 % critical value we cannot reject normality — the power law fits the data adequately. Shuffled curves typically fit better (lower AD statistic) because shuffling removes local vocabulary bursts at chapter and topic transitions: in the original order, an author introducing a new setting or character cluster causes a sudden jump in vocabulary that deviates from smooth Heaps' growth. Shuffling distributes those words uniformly, restoring the theoretical smoothness.

**Comparing β:** A higher β means faster relative vocabulary growth per additional token. Dostoevsky's philosophically and psychologically dense prose — rich with interior monologue, theological argument, and varied character voices — typically yields a higher β than Tolstoy's expansive but lexically more repetitive narrative style. The gap between original and shuffled β values quantifies how much local topic/chapter structure slows the global growth rate.

---

## A.4 Zipf's Law Analysis

### Rank-Frequency Curves

Reusing the `Counter`-based frequency list pipeline from Task 2 of the main assignment. Both novels are plotted on the same log–log axes.

In [ ]:
def rank_frequency(tokens):
    freq = Counter(tokens)
    freqs = np.array(sorted(freq.values(), reverse=True), dtype=float)
    ranks = np.arange(1, len(freqs) + 1, dtype=float)
    return ranks, freqs


ranks_tol, freqs_tol = rank_frequency(tokens_tolstoy)
ranks_dos, freqs_dos = rank_frequency(tokens_dostoevsky)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(ranks_tol, freqs_tol, color='steelblue', linewidth=0.9, label='Tolstoy')
ax.loglog(ranks_dos, freqs_dos, color='darkorange', linewidth=0.9, label='Dostoevsky')
ax.set_xlabel('Rank (log scale)')
ax.set_ylabel('Frequency (log scale)')
ax.set_title("Zipf's Law — Rank vs. Frequency")
ax.legend()
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

### Power-Law Fitting

Fit $\log f = -\alpha \log r + \log C$ via `numpy.polyfit`. **`skip=10`** trims the top 10 ranks, which are sub-Zipfian in virtually all large natural-language corpora — function words at the very top are over-represented relative to the strict inverse-rank prediction, and including them would bias the slope estimate downward.

In [ ]:
def fit_zipf(ranks, freqs, skip=10):
    lr = np.log(ranks[skip:])
    lf = np.log(freqs[skip:])
    coef = np.polyfit(lr, lf, 1)
    alpha = -coef[0]
    C = np.exp(coef[1])
    residuals = lf - np.polyval(coef, lr)
    return C, alpha, coef, lr, lf, residuals


C_tol, a_tol, coef_tol, lr_tol, lf_tol, res_tol = fit_zipf(ranks_tol, freqs_tol)
C_dos, a_dos, coef_dos, lr_dos, lf_dos, res_dos = fit_zipf(ranks_dos, freqs_dos)

print(f"{'Novel':<15s}  {'C':>10s}  {'α':>10s}")
print('-' * 38)
print(f"{'Tolstoy':<15s}  {C_tol:>10.2f}  {a_tol:>10.4f}")
print(f"{'Dostoevsky':<15s}  {C_dos:>10.2f}  {a_dos:>10.4f}")

Overlay fitted power-law lines on the rank-frequency curves.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for ranks, freqs, coef, lr, lf, color, label in [
    (ranks_tol, freqs_tol, coef_tol, lr_tol, lf_tol, 'steelblue',  'Tolstoy'),
    (ranks_dos, freqs_dos, coef_dos, lr_dos, lf_dos, 'darkorange', 'Dostoevsky'),
]:
    ax.loglog(ranks, freqs, color=color, linewidth=0.9, label=label)
    ax.loglog(np.exp(lr), np.exp(np.polyval(coef, lr)),
              color=color, ls='--', linewidth=1.2, alpha=0.8,
              label=f'{label} fit')

ax.set_xlabel('Rank (log scale)')
ax.set_ylabel('Frequency (log scale)')
ax.set_title("Zipf's Law — Rank vs. Frequency with Power-Law Fit")
ax.legend()
ax.grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

### Anderson-Darling Test

Same AD test framework as in §A.3, applied to residuals of the Zipf fit. Full significance-level table is reported for both novels.

In [ ]:
zipf_cases = [
    ('Tolstoy',    lr_tol, lf_tol, coef_tol, res_tol),
    ('Dostoevsky', lr_dos, lf_dos, coef_dos, res_dos),
]

print(f"{'Novel':<15s}  {'AD stat':>8s}  {'5% CV':>8s}  Reject H₀?")
print('-' * 47)
for name, lr, lf, coef, res_vec in zipf_cases:
    ad_res = anderson(res_vec, dist='norm')
    idx = np.argmin(np.abs(np.array(ad_res.significance_level) - 5.0))
    reject = ad_res.statistic > ad_res.critical_values[idx]
    flag = 'yes' if reject else 'no'
    print(f"{name:<15s}  {ad_res.statistic:>8.4f}  "
          f"{ad_res.critical_values[idx]:>8.4f}  {flag}")
    print(f"  Sig. levels (%): {list(ad_res.significance_level)}")
    print(f"  Critical values: {[round(v, 4) for v in ad_res.critical_values]}")

**Interpretation.** Ideal Zipf α ≈ 1 applies to English; Russian morphology (six noun cases, aspect pairs, gender agreement) inflects each lemma into multiple surface forms, dispersing frequency across more unique types and pushing α slightly above 1. If the AD test rejects normality, the power law is insufficient: both tails deviate systematically — the sub-Zipfian head (top function words exceed the predicted rank-frequency) and the super-Zipfian tail (hapax accumulation is faster than predicted). The `skip=10` trimming removes the most egregious head deviation; residual tail deviation in large corpora is unavoidable without a more flexible model (e.g., the Mandelbrot law).

---

## A.5 Characteristic Words (TF-IDF)

### TF-IDF Matrix

Each novel is treated as a single document (2-document corpus). **`min_df=2`** as specified: with exactly 2 documents, this requires a token to appear in *both* novels — so words unique to one author are excluded, and TF-IDF differences reflect how much more one author uses each shared word. Words exclusive to a single novel receive zero IDF weight (IDF = log(2/2) = 0) and are correctly filtered.

**`sublinear_tf=True`**: $\text{tf} = 1 + \log(\text{tf})$, damping very frequent function words that were not stopword-removed.

**`token_pattern=r'(?u)[а-яёa-z]+'`**: matches the same Cyrillic + Latin regex used throughout preprocessing.

In [ ]:
corpus = [' '.join(tokens_tolstoy), ' '.join(tokens_dostoevsky)]
authors = ['Tolstoy', 'Dostoevsky']

vectorizer = TfidfVectorizer(
    min_df=2,
    sublinear_tf=True,
    token_pattern=r'(?u)[а-яёa-z]+',
)
tfidf_matrix = vectorizer.fit_transform(corpus)
vocab = np.array(vectorizer.get_feature_names_out())

print(f"Shared vocabulary (min_df=2): {len(vocab):,}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

### Top-30 Words per Author

In [ ]:
rows = []
for i, author in enumerate(authors):
    scores = tfidf_matrix[i].toarray().flatten()
    top_idx = np.argsort(scores)[::-1][:30]
    for rank, idx in enumerate(top_idx, 1):
        rows.append({
            'Author': author,
            'Rank': rank,
            'Word': vocab[idx],
            'TF-IDF': round(float(scores[idx]), 4),
        })

top_df = pd.DataFrame(rows)

tol_top = (top_df[top_df['Author'] == 'Tolstoy']
           .drop(columns='Author')
           .reset_index(drop=True)
           .rename(columns={'Word': 'Tolstoy word', 'TF-IDF': 'Tolstoy TF-IDF'}))

dos_top = (top_df[top_df['Author'] == 'Dostoevsky']
           .drop(columns='Author')
           .reset_index(drop=True)
           .rename(columns={'Word': 'Dos. word', 'TF-IDF': 'Dos. TF-IDF'}))

pd.concat([tol_top, dos_top], axis=1).drop(columns='Rank')